# Lesson 2: Prompt caching, and how to break it without noticing

*Module 2 · about 10 minutes · API key required*

Prompt caching is the easiest large saving in this course. Every major provider offers it, and it often takes less than an hour to switch on. Plenty of teams "turn it on" and never actually get a cache hit, though, because something in the prompt changes on every call. Nothing errors when that happens. The bill just doesn't go down.

In this notebook we'll make a cache write, then a cache hit, then break the cache on purpose with a timestamp, and then fix it. After that we'll work out when caching pays for itself and what it really does to a total bill.

By the end you should be able to:

1. Read the cache fields in the usage object for both OpenAI and Anthropic, and avoid double-counting them.
2. Spot the most common way caching gets silently broken, and fix it.
3. Estimate the real saving on your bill, which is usually a lot less than the "90% off" headline.


### How prompt caching works

When a model reads your prompt, it turns every token into internal state (the attention "keys and values") before it can write anything. For a long system prompt that's real work, and without caching it's redone from scratch on every call even when the text is identical.

With prompt caching, the provider keeps that processed state for a few minutes. If your next request **starts with exactly the same tokens**, it reuses the stored state for that part and only processes the rest. You're billed a fraction of the normal input price for the reused part.

Three details matter:

- **It's a prefix match.** The cache covers the prompt from the first token up to the first difference. Change one character near the top and everything after it is a miss. That's why the order of things in your prompt matters so much.
- **There's a minimum size.** Prompts shorter than about 1,024 tokens (more on some models; Anthropic's Haiku 4.5 needs 4,096) aren't cached at all, and you get no error, just no cache.
- **Entries expire.** Anthropic's default cache lives 5 minutes, and every hit resets the timer. You can pay extra for a 1-hour lifetime. OpenAI's cache is automatic and best-effort, usually kept for minutes, sometimes longer.

The two vendors also *bill* it differently, which matters when you build dashboards:

| | Anthropic | OpenAI |
|---|---|---|
| How you turn it on | Mark the block with `cache_control` | Automatic for prompts over 1,024 tokens |
| Writing the cache | Costs 1.25× normal input (2× for the 1-hour option) | No extra charge |
| Reading the cache | About 0.1× normal input | About 0.1× on current models (older ones 0.25–0.5×) |
| Usage fields | `input_tokens`, `cache_creation_input_tokens`, `cache_read_input_tokens`, all separate | `prompt_tokens` **includes** `prompt_tokens_details.cached_tokens` |

That last row is a classic reporting bug. With OpenAI the cached tokens are already inside `prompt_tokens`, so if you add them again you double-count. `coursekit` normalises both vendors to the same four buckets (fresh input, cache write, cache read, output) so the numbers in this notebook are comparable.


### How these notebooks work

Run the cells in order, top to bottom. Before each code cell there's a short explanation of what it does and what to look at in the output. After the important ones there's a note on how to read what you got. Your numbers won't match mine exactly, because models are non-deterministic and prices change, so the notes describe what to look for rather than quoting fixed values.

A few conventions:

- **In class:** notes are cues for when we run this together. If you're working alone, just read them as a prompt to stop and think.
- Every notebook that spends money ends with a **ledger**: one row per API call and the total you spent.
- The **Check yourself** questions at the end have answers hidden under a click. Try them before you look.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, vendor_tokens, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
LIVE = cfg.live


  Provider : anthropic
  floor    : claude-haiku-4-5
  mid      : claude-sonnet-5
  frontier : claude-opus-5
  Cache    : explicit cache_control; read/write are separate buckets.
Switch with LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env
  Rate card: verified 5 Sep 2026 — re-check before presenting.


That cell reads your `.env`, picks OpenAI or Anthropic depending on which key it finds, and prints the three model tiers the notebook will use (floor, mid, frontier).

If the banner names a provider, the live cells will make real calls. Every lesson costs cents, not dollars. If it says `offline`, all the arithmetic still runs, but cells that need a model's answer print a placeholder and tell you they can't draw a conclusion. You can read an offline run, but it's no substitute for a live one in the caching, compression, and routing lessons.

To switch vendors, set `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and run the cell again.


---
## A long, stable system prompt

We need something worth caching, so we'll build a big block of company policy: about 5,000 tokens of repetitive support rules for a made-up logistics company. That's comfortably above every model's minimum, including Haiku's 4,096.


In [2]:
import datetime, time

POLICY_BLOCK = (
    "You are a support assistant for Northwind Logistics.\n\n"
    + "\n".join(
        f"POLICY {i:03d}: Refund requests under $500 may be approved without escalation when the "
        f"shipment was delayed more than 48 hours and the customer has fewer than three prior claims "
        f"in the trailing twelve months. Document the decision under case note code RF-{i:03d}."
        for i in range(1, 100)
    )
)
print(f"policy block: about {ntok(POLICY_BLOCK):,} tokens (tiktoken estimate)")
print(f"provider: {cfg.provider}   model: {MODELS.mid}")


policy block: about 5,059 tokens (tiktoken estimate)
provider: anthropic   model: claude-sonnet-5


---
## Call 1: the cold cache

We ask one question with the policy block as the system prompt and caching switched on. This prefix hasn't been seen before, so there's nothing to reuse yet.

What you should see depends on the provider:

- **Anthropic:** `cache_creation_input_tokens` is about the size of the policy block and `cache_read_input_tokens` is 0. You paid the 1.25× write premium on those tokens.
- **OpenAI:** `cached_tokens` is 0. There's no write premium, so this call costs the same as it would without caching.

The first call is supposed to look a little *more* expensive on Anthropic. That's the cache being filled.

You may also notice that Anthropic counts the policy block as quite a lot more tokens than the tiktoken estimate above. That's the tokenizer difference from Lesson 1: the same text, counted by a different tokenizer.


In [3]:
r1 = complete(
    "What is the refund threshold?",
    system=POLICY_BLOCK,
    model=MODELS.mid,
    max_tokens=120,
    cache=True,
    label="1. cold cache (write)",
)
print("\nraw usage object from the provider:")
print(getattr(r1.raw, "usage", "(none: offline placeholder)"))


1. cold cache (write)                         $0.002794   in=13      out=120    cw=0       cr=7841    CACHE HIT

raw usage object from the provider:
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=7841, inference_geo='global', input_tokens=13, output_tokens=120, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard')


---
## Call 2: a different question, same prefix

Now a *different* user question, but the system prompt is byte-for-byte the same as call 1. The cache key is the prefix, not the question, so this should be a hit.

Look at `cache_read` in the ledger line and in the raw usage. It should be close to the size of the policy block, and the cost of the call should drop.


In [4]:
time.sleep(1)
r2 = complete(
    "How many prior claims disqualify a customer?",
    system=POLICY_BLOCK,
    model=MODELS.mid,
    max_tokens=120,
    cache=True,
    label="2. warm cache (read)",
)
print("\nraw usage object from the provider:")
print(getattr(r2.raw, "usage", "(none: offline placeholder)"))
print()
if r2.fallback:
    print("Offline placeholder, so there is no cache behaviour to look at.")
elif r2.cache_read:
    print(f"Cache hit: {r2.cache_read:,} tokens read from cache. "
          f"This call cost {usd(r2.usd)} against {usd(r1.usd)} for call 1.")
else:
    print("No cache read this time. On OpenAI the cache is best-effort, so try running the cell again. "
          "If it never hits, the prefix may be under the model's minimum.")


2. warm cache (read)                          $0.002804   in=18      out=120    cw=0       cr=7841    CACHE HIT

raw usage object from the provider:
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=7841, inference_geo='global', input_tokens=18, output_tokens=120, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard')

Cache hit: 7,841 tokens read from cache. This call cost $0.002804 against $0.002794 for call 1.


---
## Breaking it: a timestamp at the top

This is the mistake that shows up in real codebases. Someone adds the current time to the top of the system prompt, maybe for logging or so the model knows today's date. A request ID, a session ID, or the user's name at the top does the same thing.

Now the first few tokens differ on every call, and because caching is a prefix match, nothing after them can be reused. Watch the cache read go back to zero. On Anthropic the cache write comes back too, so you pay the 1.25× premium on every call and never get a hit to pay it back.


In [5]:
BAD_PREFIX = f"Session started: {datetime.datetime.now().isoformat()}\n\n" + POLICY_BLOCK
r3 = complete(
    "What is the refund threshold?",
    system=BAD_PREFIX,
    model=MODELS.mid,
    max_tokens=120,
    cache=True,
    label="3. timestamp at the front",
)
print()
if r3.fallback:
    print("Offline placeholder, so there is no cache behaviour to look at.")
elif r3.cache_read == 0:
    print("No cache read. The API returned a normal answer with no error or warning, "
          "and the policy block was billed at full price"
          + (" plus the write premium." if r3.cache_write else "."))
else:
    print(f"Unexpected: {r3.cache_read:,} tokens were still read from cache. "
          "Some providers cache in fixed-size chunks, so check where the first difference falls.")


3. timestamp at the front                       $0.0209   in=13      out=120    cw=7862    cr=0       

No cache read. The API returned a normal answer with no error or warning, and the policy block was billed at full price plus the write premium.


### The fix: put anything that changes at the end

Same timestamp, same policy, same question. The only change is that the timestamp now goes at the end of the *user* message, after the cached prefix. Look for the cache read coming back.

A general ordering rule that holds up well: **tool definitions → system prompt → stable reference material → retrieved documents → conversation history → the new user message**. Things that never change go first, and things that change every call go last.


In [6]:
r4 = complete(
    f"What is the refund threshold?\n\n[session {datetime.datetime.now().isoformat()}]",
    system=POLICY_BLOCK,
    model=MODELS.mid,
    max_tokens=120,
    cache=True,
    label="4. timestamp at the end",
)
print()
if r4.fallback:
    print("Offline placeholder, so there is no cache behaviour to look at.")
elif r4.cache_read:
    print(f"Cache hit again: {r4.cache_read:,} tokens read from cache. "
          "The model got the same information, just in a different position.")
else:
    print("No hit this time. If call 2 did hit, the cache may have expired or (on OpenAI) "
          "landed on a different server. Re-run the cell.")


4. timestamp at the end                       $0.002834   in=33      out=120    cw=0       cr=7841    CACHE HIT

Cache hit again: 7,841 tokens read from cache. The model got the same information, just in a different position.


---
## When does caching pay for itself?

On Anthropic the first call costs more (the write premium) and later calls cost less (the read discount). So how many calls does it take to come out ahead? Without caching, $N$ calls cost $N \cdot P_{in}$. With caching, they cost $P_{write} + (N-1) \cdot P_{read}$. Set the two equal and solve for $N$:

$$N_{\text{break-even}} = \frac{P_{write} - P_{read}}{P_{in} - P_{read}}$$

The cell below works this out from the rate card. It also shows Anthropic's 1-hour cache option, where the write costs 2× instead of 1.25×. For OpenAI models the write costs nothing extra, so the answer is 1: caching can't lose money.


In [7]:
rows = []
for m in ["claude-haiku-4-5", "claude-sonnet-5", "claude-opus-5", "claude-fable-5-1",
          "gpt-5.6-luna", "gpt-5.6-terra"]:
    p = PRICES[m]
    rows.append(dict(model=m, ttl="5 min" if m.startswith("claude") else "auto",
                     write_x=p["cw"] / p["inp"], read_x=p["cr"] / p["inp"],
                     break_even_calls=cache_breakeven(m)))
    if m == "claude-sonnet-5":   # the 1-hour option doubles the write price
        w, r, i = 2 * p["inp"], p["cr"], p["inp"]
        rows.append(dict(model=m, ttl="1 hour", write_x=2.0, read_x=r / i,
                         break_even_calls=(w - r) / (i - r)))
show(pd.DataFrame(rows).style.format({"write_x": "{:.2f}x", "read_x": "{:.3f}x",
                                      "break_even_calls": "{:.2f}"}))


,model,ttl,write_x,read_x,break_even_calls
0,claude-haiku-4-5,5 min,1.25x,0.100x,1.28
1,claude-sonnet-5,5 min,1.25x,0.100x,1.28
2,claude-sonnet-5,1 hour,2.00x,0.100x,2.11
3,claude-opus-5,5 min,1.25x,0.100x,1.28
4,claude-fable-5-1,5 min,1.25x,0.025x,1.26
5,gpt-5.6-luna,auto,1.00x,0.100x,1.00
6,gpt-5.6-terra,auto,1.00x,0.100x,1.00


With the 5-minute cache, break-even lands between call 1 and call 2, so the second call already puts you ahead. With the 1-hour cache it takes a third call. For any prompt prefix that's reused more than a couple of times, caching wins. The practical problem is never whether caching is worth it. It's making sure the cache actually hits.


---
## What caching really does to the whole bill

"90% off" is true, but only for the *cached input tokens*. Your bill also has uncached input and output, and output is never cached and is the expensive side. So the saving on the total bill is roughly:

$$\text{saving} \approx \text{hit rate} \times (1 - \tfrac{P_{read}}{P_{in}}) \times \text{input's share of the bill}$$

Input's share of the bill is the part people forget, and it varies a lot between workloads. The cell below sweeps the hit rate for two workloads on the mid-tier model:

- **RAG-style**: 8,000 input tokens (lots of retrieved documents) and 400 output. Input dominates the bill.
- **Chat-style**: 1,500 input tokens and 400 output. Output is more than half the bill.


In [8]:
def bill(model, tok_in, tok_out, hit_rate):
    cached = tok_in * hit_rate
    return cost(model, inp=tok_in - cached, out=tok_out, cache_r=cached)

M = MODELS.mid
workloads = {"RAG-style (8000 in / 400 out)": (8000, 400),
             "chat-style (1500 in / 400 out)": (1500, 400)}
rows = []
for name, (tin, tout) in workloads.items():
    base = bill(M, tin, tout, 0.0)
    input_share = cost(M, inp=tin) / base
    for h in [0.0, 0.5, 0.8, 0.95]:
        rows.append(dict(workload=name, input_share=input_share, hit_rate=h,
                         cost_per_call=bill(M, tin, tout, h),
                         saving=1 - bill(M, tin, tout, h) / base))
df = pd.DataFrame(rows)
show(df.style.format({"input_share": "{:.0%}", "hit_rate": "{:.0%}",
                      "cost_per_call": "${:,.5f}", "saving": "{:.0%}"}))


,workload,input_share,hit_rate,cost_per_call,saving
0,RAG-style (8000 in / 400 out),80%,0%,$0.02000,0%
1,RAG-style (8000 in / 400 out),80%,50%,$0.01280,36%
2,RAG-style (8000 in / 400 out),80%,80%,$0.00848,58%
3,RAG-style (8000 in / 400 out),80%,95%,$0.00632,68%
4,chat-style (1500 in / 400 out),43%,0%,$0.00700,0%
5,chat-style (1500 in / 400 out),43%,50%,$0.00565,19%
6,chat-style (1500 in / 400 out),43%,80%,$0.00484,31%
7,chat-style (1500 in / 400 out),43%,95%,$0.00443,37%


**Reading the output.** At an 80% hit rate the RAG-style workload saves close to 60% of its bill, because input is most of what it pays for. The chat-style workload saves only about 30%, because most of its bill is output, which caching can't touch.

So when someone asks how much caching will save, the honest answer is: it depends on how much of your bill is input. Work out that share from your real usage data before you promise a number. For agents and RAG it's usually high. For chat with long answers it's usually lower.


In [9]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.0293


,label,model,input,output,cache_write,cache_read,usd,note
0,1. cold cache (write),claude-sonnet-5,13,120,0,7841,0.002794,CACHE HIT
1,2. warm cache (read),claude-sonnet-5,18,120,0,7841,0.002804,CACHE HIT
2,3. timestamp at the front,claude-sonnet-5,13,120,7862,0,0.020881,
3,4. timestamp at the end,claude-sonnet-5,33,120,0,7841,0.002834,CACHE HIT


---
## What to take away

- Caching is mostly about prompt order. Put static content first and anything that changes per request last.
- A timestamp, request ID, or user name at the top of a prompt quietly disables caching, with no error.
- Check the cache-read field in production: `cache_read_input_tokens` on Anthropic, `prompt_tokens_details.cached_tokens` on OpenAI. If it's near zero, you have this bug.
- On OpenAI, cached tokens are already inside `prompt_tokens`. Don't add them twice.
- The saving on the total bill is the input discount times input's share of the bill. Measure that share before you quote a figure.


### Check yourself

**1. Your system prompt starts with `You are helping {customer_name}.` followed by 6,000 tokens of fixed instructions. What happens to caching, and how would you fix it?**

<details><summary>Show answer</summary>

The customer name differs between customers, so the prefix differs from the very first line and the 6,000 tokens after it can't be reused across customers. Move the fixed instructions to the top and put the customer name in the user message (or at the end of the system prompt).

</details>

**2. On OpenAI, a response reports `prompt_tokens = 5,200` and `cached_tokens = 5,000`. How many input tokens were billed at the full price?**

<details><summary>Show answer</summary>

**200.** The 5,000 cached tokens are already included in the 5,200. Billing them as 5,200 full-price plus 5,000 cached would double-count.

</details>

**3. Your workload is 1,000 input and 1,000 output tokens per call on a model where output costs 5× input. Roughly what's the best-case saving from caching, even at a 100% hit rate?**

<details><summary>Show answer</summary>

Input is 1,000 × 1 = 1,000 units and output is 1,000 × 5 = 5,000 units, so input is about 17% of the bill. Even with every input token cached at a 90% discount, you save about 0.9 × 17% ≈ **15%**.

</details>


### Try it on your own work

Take a sample of 100 production calls and log the cache-read field for each one. If the hit rate is close to zero, search your prompt templates for anything dynamic near the top: timestamps, request or trace IDs, user or tenant names, and JSON built from a dictionary whose key order isn't fixed.
